In [1]:
import pandas as pd
import icartt
import os
import warnings
import re
from datetime import datetime
import csv
from datetime import datetime, timedelta
from netCDF4 import Dataset
import numpy as np
from scipy import stats
import glob
from math import pi
import ast

In [2]:
import pandas as pd
import numpy as np
import os

# Define organizations and paths
ORGANIZATIONS = ["NASA", "DOE", "NSF", "NOAA"]

NSF_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\NSF_restricted.csv"
NOAA_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\NOAA_restricted.csv"
NASA_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\NASA_restricted.csv"
DOE_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\DOE_restricted.csv"
MASTER_RESTRICTED_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\restricted_combined\master_restricted.csv"

NSF_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\NSF_comprehensive.csv"
NOAA_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\NOAA_comprehensive.csv"
NASA_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\NASA_comprehensive.csv"
DOE_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\DOE_comprehensive.csv"
MASTER_COMPREHENSIVE_PATH = rf"C:\Users\haika\Desktop\May_Research\may_datasets\comprehensive_combined\master_comprehensive.csv"

# List of all file paths
all_paths = [
    NSF_RESTRICTED_PATH,
    NOAA_RESTRICTED_PATH,
    NASA_RESTRICTED_PATH,
    DOE_RESTRICTED_PATH,
    MASTER_RESTRICTED_PATH,
    NSF_COMPREHENSIVE_PATH,
    NOAA_COMPREHENSIVE_PATH,
    NASA_COMPREHENSIVE_PATH,
    DOE_COMPREHENSIVE_PATH,
    MASTER_COMPREHENSIVE_PATH
]

def remove_empty_rows_and_save(file_path):
    """
    Remove rows that are entirely empty except for Organization, Campaign, UTC, Date columns.
    These columns are always filled so we exclude them from the empty row analysis.
    """
    try:
        print(f"\nProcessing: {os.path.basename(file_path)}")
        
        # Check if file exists
        if not os.path.exists(file_path):
            print(f"  ERROR: File not found - {file_path}")
            return
        
        # Read the CSV file
        df = pd.read_csv(file_path)
        original_rows = len(df)
        print(f"  Original rows: {original_rows:,}")
        
        # Define columns to exclude from empty row analysis
        exclude_columns = ['Organization', 'Campaign', 'UTC', 'Date']
        
        # Get columns to analyze (all columns except the excluded ones)
        analyze_columns = [col for col in df.columns if col not in exclude_columns]
        
        print(f"  Analyzing {len(analyze_columns)} columns (excluding: {', '.join(exclude_columns)})")
        
        # Create subset dataframe with only the columns we want to analyze
        df_subset = df[analyze_columns]
        
        # Find rows where ALL analyzed columns are NaN
        empty_rows_mask = df_subset.isna().all(axis=1)
        
        # Remove rows that are entirely empty in the analyzed columns
        df_cleaned = df[~empty_rows_mask]
        cleaned_rows = len(df_cleaned)
        removed_rows = original_rows - cleaned_rows
        
        print(f"  Cleaned rows: {cleaned_rows:,}")
        print(f"  Removed rows: {removed_rows:,}")
        
        if removed_rows > 0:
            # Save the cleaned dataframe back to the original path
            df_cleaned.to_csv(file_path, index=False)
            print(f"  ✓ Saved cleaned data to: {os.path.basename(file_path)}")
        else:
            print(f"  ✓ No empty rows found, file unchanged")
            
    except Exception as e:
        print(f"  ERROR processing {file_path}: {str(e)}")

# Main execution
print("REMOVING ENTIRELY EMPTY ROWS FROM ALL DATASETS")
print("=" * 60)

total_original_rows = 0
total_cleaned_rows = 0
files_processed = 0
files_with_errors = 0

# Process each file
for file_path in all_paths:
    try:
        # Get original row count
        if os.path.exists(file_path):
            df_temp = pd.read_csv(file_path)
            original_count = len(df_temp)
            
            # Remove empty rows and save
            remove_empty_rows_and_save(file_path)
            
            # Get cleaned row count
            df_temp_cleaned = pd.read_csv(file_path)
            cleaned_count = len(df_temp_cleaned)
            
            total_original_rows += original_count
            total_cleaned_rows += cleaned_count
            files_processed += 1
        else:
            print(f"\nSkipping: {os.path.basename(file_path)} (file not found)")
            
    except Exception as e:
        print(f"\nERROR with {os.path.basename(file_path)}: {str(e)}")
        files_with_errors += 1

# Summary
print(f"\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print(f"Files processed successfully: {files_processed}")
print(f"Files with errors: {files_with_errors}")
print(f"Total original rows: {total_original_rows:,}")
print(f"Total cleaned rows: {total_cleaned_rows:,}")
print(f"Total rows removed: {(total_original_rows - total_cleaned_rows):,}")
print(f"Data reduction: {((total_original_rows - total_cleaned_rows) / total_original_rows * 100):.2f}%")
print("=" * 60)

REMOVING ENTIRELY EMPTY ROWS FROM ALL DATASETS

Processing: NSF_restricted.csv
  Original rows: 49,575
  Analyzing 28 columns (excluding: Organization, Campaign, UTC, Date)
  Cleaned rows: 45,724
  Removed rows: 3,851
  ✓ Saved cleaned data to: NSF_restricted.csv

Processing: NOAA_restricted.csv
  Original rows: 462,480
  Analyzing 28 columns (excluding: Organization, Campaign, UTC, Date)
  Cleaned rows: 462,480
  Removed rows: 0
  ✓ No empty rows found, file unchanged

Processing: NASA_restricted.csv
  Original rows: 4,712,873
  Analyzing 28 columns (excluding: Organization, Campaign, UTC, Date)
  Cleaned rows: 4,710,510
  Removed rows: 2,363
  ✓ Saved cleaned data to: NASA_restricted.csv

Processing: DOE_restricted.csv
  Original rows: 1,516,803
  Analyzing 28 columns (excluding: Organization, Campaign, UTC, Date)
  Cleaned rows: 1,469,190
  Removed rows: 47,613
  ✓ Saved cleaned data to: DOE_restricted.csv


C:\Users\haika\AppData\Local\Temp\ipykernel_13124\3461304501.py:98: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(file_path)



Processing: master_restricted.csv


C:\Users\haika\AppData\Local\Temp\ipykernel_13124\3461304501.py:48: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  Original rows: 4,790,562
  Analyzing 35 columns (excluding: Organization, Campaign, UTC, Date)
  Cleaned rows: 4,790,562
  Removed rows: 0
  ✓ No empty rows found, file unchanged


...


Processing: master_comprehensive.csv


C:\Users\haika\AppData\Local\Temp\ipykernel_13124\3461304501.py:48: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file_path)


  Original rows: 6,832,843
  Analyzing 54 columns (excluding: Organization, Campaign, UTC, Date)
  Cleaned rows: 6,832,843
  Removed rows: 0
  ✓ No empty rows found, file unchanged


C:\Users\haika\AppData\Local\Temp\ipykernel_13124\3461304501.py:105: DtypeWarning: Columns (3) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp_cleaned = pd.read_csv(file_path)



SUMMARY
Files processed successfully: 10
Files with errors: 0
Total original rows: 25,231,397
Total cleaned rows: 25,144,152
Total rows removed: 87,245
Data reduction: 0.35%
